<h1 style="text-align:center; color: #2E86C1; font-family:Arial;"> DL.AI - Pydantic for LLM Workflows </h1>

In [32]:
# !pip install -U -q "google"
# !pip install -U -q "google.genai"

import os
from google.colab import userdata
from google.colab import drive
os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")

drive.mount("/content/drive")

Mounted at /content/drive


In [33]:
# Create the .ssh directory if it doesn't exist
!mkdir -p ~/.ssh

# Copy the private and public keys from Google Drive
!cp '/content/drive/MyDrive/Colab_SSH_Keys/id_rsa' ~/.ssh/id_rsa
!cp '/content/drive/MyDrive/Colab_SSH_Keys/id_rsa.pub' ~/.ssh/id_rsa.pub

# Set strict permissions (crucial for SSH)
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_rsa
!chmod 644 ~/.ssh/id_rsa.pub

# Add GitHub to known_hosts to prevent host key verification issues
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

# github.com:22 SSH-2.0-ae0a932
# github.com:22 SSH-2.0-ae0a932
# github.com:22 SSH-2.0-ae0a932
# github.com:22 SSH-2.0-ae0a932
# github.com:22 SSH-2.0-ae0a932


In [34]:
!git clone -b pydantic-for-llm-workflows git@github.com:YashwanthMRamachandra/GenAI-and-AgenticAI-Practice.git

Cloning into 'GenAI-and-AgenticAI-Practice'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (80/80), done.
Receiving objects: 100% (100/100), 103.42 KiB | 9.40 MiB/s, done.
remote: Total 100 (delta 29), reused 78 (delta 10), pack-reused 0 (from 0)
Resolving deltas: 100% (29/29), done.


# Lesson 2: Pydantic Basics

In this lesson, you'll learn the fundamentals of Pydantic models for data validation using a customer support system as your example application. You'll see how to define data models, validate user input, and handle validation errors gracefully.

By the end of this lesson, you'll be able to:
- Create Pydantic models to validate user input data
- Handle validation errors with proper error handling
- Use optional fields and field constraints in your models
- Work with JSON data validation methods

---

In [9]:
# !pip install pydantic[email]

In [8]:
# Import libraries needed for the lesson
from pydantic import BaseModel, ValidationError, EmailStr
import json

In [21]:
# @title # Create a Pydantic model for validating user input
class UserInput(BaseModel):
  name: str
  email: EmailStr
  query: str

In [22]:
# @title # Create a model instance
user_input = UserInput(
    name="yash",
    email="yash.pydantic@email.com",
    query="I forgot my password"
)
print(user_input)

name='yash' email='yash.pydantic@email.com' query='I forgot my password'


In [12]:
# @title # Attempt to create another model instance with an invalid email
user_input_invalid_email = UserInput(
    name="yash",
    email="not-an-email",
    query="I forgot my password"
)
print(user_input_invalid_email)

ValidationError: 1 validation error for UserInput
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='not-an-email', input_type=str]

In [13]:
# @title # Define a function for error handling and try different inputs
def validate_user_input(input_data):
  try:
    user_input = UserInput(**input_data)
    print(f"✅ Valid user input created:")
    print(f"{user_input.model_dump_json(indent=2)}")
    return user_input
  except ValidationError as e:
    # Capture and display validation errors in a readable format
    print(f"❌ Validation error occurred:")
    for error in e.errors():
      print(f"  - {error['loc'][0]}: {error['msg']}")
      print(f"Current Input/s:")
      print(json.dumps(error["input"], indent=2))
    return None

In [14]:
input_data = {
    "name": "yash",
    "email": "yash.pydantic@email.com",
    "query": "I forgot my password"
}

user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "yash",
  "email": "yash.pydantic@email.com",
  "query": "I forgot my password"
}


In [15]:
# Attempt to create an instance of UserInput with missing query field
input_data = {
    "name": "Joe User",
    "email": "joe.user@example.com"
}

user_input = validate_user_input(input_data)

❌ Validation error occurred:
  - query: Field required
Current Input/s:
{
  "name": "Joe User",
  "email": "joe.user@example.com"
}


In [44]:
# @title # Update your UserInput data model with additional fields and experiment with different input data
from pydantic import Field
from typing import Optional
from datetime import date

class UserInput(BaseModel):
  name: str
  email: EmailStr
  query: str
  order_id: Optional[int] = Field(
      None,
      description="5-digit order number (cannot start with 0)",
      ge=10000,
      le=99999
  )
  purchase_date: Optional[date] = None

In [45]:
# Define a dictionary with required fields only
input_data = {
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": "I forgot my password."
}

user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I forgot my password.",
  "order_id": null,
  "purchase_date": null
}


In [46]:
# Define a dictionary with all fields including optional ones
input_data = {
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": f"""I bought a laptop carrying case and it turned out to be
             the wrong size. I need to return it.""",
    "order_id": 12345,
    "purchase_date": date(2025, 12, 31)
}

In [47]:
user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be \n             the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}


In [48]:
# Define a dictionary with all fields and including additional ones
input_data = {
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": f"""I bought a laptop carrying case and it turned out to be
             the wrong size. I need to return it.""",
    "order_id": 12345,
    "purchase_date": date(2025, 12, 31),
    "system_message": "logging status regarding order processing...",
    "iteration": 1
}

user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be \n             the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}


In [49]:
# Create an instance of UserInput with valid data
input_data = {
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": f"""I bought a laptop carrying case and it turned out to be
             the wrong size. I need to return it.""",
    "order_id": 12345,
    "purchase_date": "2025-12-31"
}

user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be \n             the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}


In [50]:
# Define order_id as a string
input_data = {
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": f"""I bought a laptop carrying case and it turned out to be
             the wrong size. I need to return it.""",
    "order_id": "12345",
    "purchase_date": "2025-12-31"
}

user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be \n             the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}


In [51]:
# Define name field as an integer
input_data = {
    "name": 99999,
    "email": "joe.user@example.com",
    "query": f"""I bought a laptop carrying case and it turned out to be
             the wrong size. I need to return it.""",
    "order_id": 12345,
    "purchase_date": "2025-12-31"
}

user_input = validate_user_input(input_data)
print(user_input)

❌ Validation error occurred:
  - name: Input should be a valid string
Current Input/s:
99999
None


In [52]:
# @title # Try starting with JSON data as input
# Define user input as JSON data
json_data = '''
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I bought a keyboard and mouse and was overcharged",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}
'''

# Parse the JSON string into a Python dict
input_data = json.loads(json_data)
user_input = validate_user_input(input_data)

✅ Valid user input created:
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I bought a keyboard and mouse and was overcharged",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}


In [78]:
# Try different JSON input
json_data = '''
{
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": "My account has been locked for some reason.",
    "order_id": "01234",
    "purchase_date": "2025-12-31"
}
'''

In [79]:
input_data = json.loads(json_data)
user_input = validate_user_input(input_data)

❌ Validation error occurred:
  - order_id: Input should be greater than or equal to 10000
Current Input/s:
"01234"


In [80]:
# @title # Parse JSON and validate user input data in one step using model_validate_json method
user_input = UserInput.model_validate_json(json_data)
print(user_input)

ValidationError: 1 validation error for UserInput
order_id
  Input should be greater than or equal to 10000 [type=greater_than_equal, input_value='01234', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal

In [81]:
# Try different JSON input
json_data = '''
{
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": "My account has been locked for some reason.",
    "order_id": 12345,
    "purchase_date": "2025-12-31"
}
'''

In [86]:
user_input = UserInput.model_validate_json(json_data)
print(user_input)

name='Yash User' email='yash.user@example.com' query='My account has been locked for some reason.' order_id=12345 purchase_date=datetime.date(2025, 12, 31)


__main__.UserInput

# Lesson 3: Prompting for structure and setting up a retry method

In this notebook, you'll learn how to combine Pydantic with retry strategies to reliably extract structured output from an LLM.

By the end, you'll be able to:
- Define structured data models for LLM responses
- Build robust retry mechanisms for validation errors
- Create reusable functions for LLM interactions

---

### Import packages and initialize the OpenAI client

In [11]:
# !pip install pydantic[email]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.9 MB/s eta 0:00:00


In [6]:
# Import necessary packages
from pydantic import BaseModel, ValidationError, Field, EmailStr
from typing import List, Literal, Optional
import json
from datetime import date
# import openai
import google.generativeai as openai
import os

In [7]:
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [27]:
# @title # Define some sample input data
# Define a JSON string representing user input
user_input_json = '''
{
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": "I want to get an update about the status of my refund.",
    "order_number": null,
    "purchase_date": null
}
'''

In [18]:
# @title # Define UserInput Model
class UserInput(BaseModel):
  name: str
  email: EmailStr
  query: str
  order_id: Optional[int] = Field(
      None,
      description="5-digit order number (cannot start with 0)",
      ge=10000,
      le=99999
  )
  purchase_date: Optional[date] = None

In [28]:
# Create UserInput instance from JSON data
user_input = UserInput.model_validate_json(user_input_json)
print(user_input)

name='Yash User' email='yash.user@example.com' query='I want to get an update about the status of my refund.' order_id=None purchase_date=None


In [20]:
# @title # Create a new data model called CustomerQuery
class CustomerQuery(UserInput):
  priority: str = Field(
      ..., description="Priority Level: High, Medium, or Low"
  )
  category: Literal[
      'refund_request', 'information_request', 'other'
  ] = Field(
      ..., description="Query Category")
  is_compliant: bool = Field(
      ..., description="Whether this is compliant"
  )
  tags: List = Field(
      ..., description="Relevant keyword tags"
  )

In [21]:
# @title # Construct a prompt with example output
example_response_structure = f"""{{
  name = 'Yash user',
  email = 'yash.user@example.com',
  query = 'I ordered a new computer monitor and it arrived a screen cracked. I need to exchange it for new one.',
  order_id = 12345,
  purchase_date = '2025-12-31',
  category = 'refund_request',
  priority = 'Medium',
  is_compliant = True,
  tags = ['monitor', 'support', 'exchange']
}}
"""

In [29]:
# @title # Create prompt with user data and expected JSON structure
prompt = f"""
Please analyze this user query \n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching this exact structure and data types:
{example_response_structure}
Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.
"""

print(prompt)


Please analyze this user query 
 {
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I want to get an update about the status of my refund.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure and data types:
{
  name = 'Yash user',
  email = 'yash.user@example.com',
  query = 'I ordered a new computer monitor and it arrived a screen cracked. I need to exchange it for new one.',
  order_id = 12345,
  purchase_date = '2025-12-31',
  category = 'refund_request',
  priority = 'Medium',
  is_compliant = True,
  tags = ['monitor', 'support', 'exchange']
}

Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.



In [24]:
# @title # Define a function to call an LLM and try it with your prompt
openai.configure(api_key=os.environ["GEMINI_API_KEY"])

In [30]:
def call_llm(prompt, model="gemini-2.5-flash"):
  """Call gemini LLM with Prompt"""
  model_instance = openai.GenerativeModel(model)
  response = model_instance.generate_content(prompt)

  return response.text

In [31]:
# Get response from LLM
response_content = call_llm(prompt)
print(response_content)

```json
{
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I want to get an update about the status of my refund.",
  "order_id": null,
  "purchase_date": null,
  "category": "refund_status_inquiry",
  "priority": "Medium",
  "is_compliant": true,
  "tags": ["refund", "status", "update"]
}
```


In [36]:
# @title # Validate the LLM output using your CustomerQuery model
# Attempt to parse the response into CustomerQuery model
validate_data = CustomerQuery.model_validate_json(response_content)
print(validate_data)

ValidationError: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n  "name": "Y...tus", "update"]\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid

In [37]:
# @title # Define a function for error handling
def validate_with_model(data_model, llm_response):
  try:
    validate_data = data_model.model_validate_json(llm_response)
    print("data validation successful!")
    print(validate_data.model_dump_json(indent=2))
    return validate_data, None
  except ValidationError as e:
    print(f"error validating data: {e}")
    error_message = (
        f"This response generated a validation error: {e}"
    )
    return None, error_message

In [38]:
validate_data, validation_error = validate_with_model(
    CustomerQuery, response_content
)

error validating data: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n  "name": "Y...tus", "update"]\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


In [39]:
from google.api_core import retry
# @title # Define a function to create a retry prompt including error details
def create_retry_prompt(original_prompt, original_response, error_message):
  retry_prompt = f"""
  This is a request to fix an error in the structure of an llm_response.
  Here is the original request:
  <original_prompt>
  {original_prompt}
  </original_prompt>

  Here is the original llm_response:
  <llm_response>
  {original_response}
  </llm_response>

  Compare the error message and the llm_response and identify what needs to be fixed or removed in the llm_response to resolve this error.

  Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.
  """

  return retry_prompt

In [40]:
validation_retry_prompt = create_retry_prompt(
    prompt, response_content, validation_error
)
print(validation_retry_prompt)


  This is a request to fix an error in the structure of an llm_response.
  Here is the original request:
  <original_prompt>
  
Please analyze this user query 
 {
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I want to get an update about the status of my refund.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure and data types:
{
  name = 'Yash user',
  email = 'yash.user@example.com',
  query = 'I ordered a new computer monitor and it arrived a screen cracked. I need to exchange it for new one.',
  order_id = 12345,
  purchase_date = '2025-12-31',
  category = 'refund_request',
  priority = 'Medium',
  is_compliant = True,
  tags = ['monitor', 'support', 'exchange']
}

Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.

  </original_prompt>

  Here is the original llm_response:
  <llm_response>
  ```json
{
  "name": "Yash 

In [41]:
# @title # Call the LLM with retry prompt
validate_retry_response = call_llm(validation_retry_prompt)
print(validate_retry_response)

```json
{
  "fix_description": "The original prompt explicitly states: 'Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.' The `llm_response` included markdown code block delimiters (` ```json ` and ` ``` `) which constitute formatting before and after the JSON object. These should be removed.",
  "changes_required": [
    {
      "type": "remove",
      "target": "start_markdown_wrapper",
      "content_to_remove": "```json"
    },
    {
      "type": "remove",
      "target": "end_markdown_wrapper",
      "content_to_remove": "```"
    }
  ],
  "fixed_llm_response_example": {
    "name": "Yash User",
    "email": "yash.user@example.com",
    "query": "I want to get an update about the status of my refund.",
    "order_id": null,
    "purchase_date": null,
    "category": "refund_status_inquiry",
    "priority": "Medium",
    "is_compliant": true,
    "tags": ["refund", "status", "update"]
  },
  "notes_on_other_d

In [42]:
validate_data, validation_error = validate_with_model(
    CustomerQuery, validate_retry_response
)

error validating data: 1 validation error for CustomerQuery
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n  "fix_descr... user query\'."\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


In [45]:
# @title # Create second retry prompt
second_validation_retry_prompt = create_retry_prompt(
    validation_retry_prompt, validate_retry_response, validation_error
)
print(second_validation_retry_prompt)


  This is a request to fix an error in the structure of an llm_response.
  Here is the original request:
  <original_prompt>
  
  This is a request to fix an error in the structure of an llm_response.
  Here is the original request:
  <original_prompt>
  
Please analyze this user query 
 {
  "name": "Yash User",
  "email": "yash.user@example.com",
  "query": "I want to get an update about the status of my refund.",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure and data types:
{
  name = 'Yash user',
  email = 'yash.user@example.com',
  query = 'I ordered a new computer monitor and it arrived a screen cracked. I need to exchange it for new one.',
  order_id = 12345,
  purchase_date = '2025-12-31',
  category = 'refund_request',
  priority = 'Medium',
  is_compliant = True,
  tags = ['monitor', 'support', 'exchange']
}

Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or 

In [46]:
# Call the LLM with the second validation retry prompt
second_validation_retry_response = call_llm(second_validation_retry_prompt)
print(second_validation_retry_response)

```json
{
  "fix_description": "The provided `llm_response` includes markdown code block delimiters (` ```json ` at the start and ` ``` ` at the end). The `original_prompt` for this task explicitly states: 'Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.' These delimiters constitute formatting and violate the specified output format. They should be removed.",
  "changes_required": [
    {
      "type": "remove",
      "target": "start_markdown_wrapper",
      "content_to_remove": "```json"
    },
    {
      "type": "remove",
      "target": "end_markdown_wrapper",
      "content_to_remove": "```"
    }
  ],
  "fixed_llm_response_example": {
    "fix_description": "The original prompt explicitly states: 'Respond ONLY with valid JSON. Do not include any explanation or other text or formatting before or after the JSON object.' The `llm_response` included markdown code block delimiters (` ```json ` and ` ``` `) whic

In [35]:
!cp '/content/drive/MyDrive/Colab Notebooks/Pydantic for LLM Workflows.ipynb' \
    '/content/GenAI-and-AgenticAI-Practice'